Data Reading

In [0]:
#%sql
#LIST 'abfss://bronze@retailstgaccpd.dfs.core.windows.net/';

path,name,size,modification_time
abfss://bronze@retailstgaccpd.dfs.core.windows.net/raw_data/,raw_data/,0,1773749310000


# Reading Raw Data

In [0]:
raw_data= spark.read.format("parquet").\
    option("inferSchema",True).\
        load("abfss://bronze@retailstgaccpd.dfs.core.windows.net/raw_data")

#Data Transformation

In [0]:
from pyspark.sql.functions import *
raw_data= raw_data.withColumn("Model_Category", split(col("Model_ID"), "-")[0])

In [0]:
raw_data= raw_data.withColumn("RevenuePerUnits", col("Revenue")/col("Units_Sold"))
#raw_data.display()

In [0]:
#from pyspark.sql.types import *
#raw_data=raw_data.withColumn("Date", make_date(col("Year").cast("StringType"), col("Month").cast("StringType"), col("Day").cast("StringType")))

#Writing Back silver data to datalake

In [0]:
raw_data.write.format("parquet").\
    mode("overwrite").\
        save("abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data/")

In [0]:
'''silver_data= spark.read.format("parquet").\
    option("inferSchema",True).\
        load("abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data")'''

In [0]:
#silver_data.groupBy("Year","BranchName").agg(sum("Units_Sold").alias("Total_Units_Sold")).sort("Year","Total_Units_Sold", ascending=[1,0]).display()


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4942383119949104>, line 1
----> 1 silver_data.groupBy("Year","BranchName").agg(sum("Units_Sold").alias("Total_Units_Sold")).sort("Year","Total_Units_Sold", ascending=[1,0]).display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:74, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     70 def df_display(df, *args, **kwargs):
     71     """
     72     df.display() is an alias for display(df). Run help(display) for more information.
     73     """
---> 74     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, Co